<div dir="rtl">

# جلسه ۹: یادگیری انتقالی (Transfer Learning) و Fine-tuning

**Transfer Learning and Fine-tuning**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/09.%20Transfer_Learning.ipynb)

**دوره یادگیری عمیق و شبکه‌های عصبی** — کیوان محمودی

### اهداف این جلسه

- درک چرایی و زمان مناسب استفاده از یادگیری انتقالی
- بارگذاری مدل‌های ازپیش‌آموزش‌دیده (Pretrained) از `torchvision.models` و پیش‌پردازش متناظر آن‌ها
- مقایسه سه راهبرد: آموزش از صفر، استخراج ویژگی (Feature Extraction) و Fine-tuning کامل
- استفاده از نرخ یادگیری متفاوت برای لایه‌های مختلف (Discriminative Learning Rates)
- تسریع آموزش با Mixed Precision

> **نکته اجرا:** این نوتبوک روی GPU سریع‌تر اجرا می‌شود. در Colab از منوی `Runtime → Change runtime type` گزینه **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl">

## ۱. ایده یادگیری انتقالی

آموزش یک ResNet روی ImageNet به ۱٫۲ میلیون تصویر برچسب‌دار و ساعت‌ها (یا روزها) محاسبه روی GPU نیاز دارد. اما در اغلب پروژه‌های واقعی فقط چند صد یا چند هزار تصویر داریم. **یادگیری انتقالی (Transfer Learning)** راه‌حل این مشکل است: مدلی را که روی یک دیتاست بزرگ آموزش دیده، به‌عنوان نقطه شروع برای مسئله خودمان استفاده می‌کنیم.

چرا کار می‌کند؟ در جلسه ۷ دیدیم که CNN ویژگی‌ها را **سلسله‌مراتبی** یاد می‌گیرد. ویژگی‌های لایه‌های اول (لبه، رنگ، بافت) تقریباً برای **هر** مسئله بینایی مفیدند و فقط لایه‌های آخر مختص مسئله اصلی (۱۰۰۰ کلاس ImageNet) هستند.

| راهبرد | چه چیزی آموزش می‌بیند؟ | مناسب برای |
|---|---|---|
| **Feature Extraction** | فقط لایه طبقه‌بند جدید؛ بقیه منجمد (Frozen) | داده بسیار کم، مسئله شبیه داده پیش‌آموزش |
| **Fine-tuning** | همه لایه‌ها، با نرخ یادگیری کوچک | داده متوسط یا مسئله متفاوت‌تر |
| **آموزش از صفر** | همه لایه‌ها از مقداردهی تصادفی | داده بسیار زیاد یا مسئله کاملاً متفاوت (مثلاً تصاویر پزشکی خاص) |

همین ایده در پردازش زبان طبیعی هم انقلاب ایجاد کرد: مدل‌هایی مثل BERT روی متن عظیم پیش‌آموزش می‌بینند و برای هر مسئله Fine-tune می‌شوند (جلسه ۱۴).

</div>

In [ ]:
import copy
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
torch.manual_seed(42)

<div dir="rtl">

## ۲. مدل‌های ازپیش‌آموزش‌دیده در torchvision

هر مدل در `torchvision.models` یک شیء `Weights` دارد که علاوه بر وزن‌ها، **پیش‌پردازش دقیقی** که مدل با آن آموزش دیده (اندازه تصویر و میانگین و انحراف معیار ImageNet) و نام کلاس‌ها را در خود دارد. استفاده از پیش‌پردازش متفاوت از زمان پیش‌آموزش، یکی از خطاهای رایج است.

</div>

In [ ]:
weights = models.ResNet18_Weights.IMAGENET1K_V1
pretrained = models.resnet18(weights=weights).to(device).eval()
preprocess = weights.transforms()
print(preprocess)
print('number of ImageNet classes:', len(weights.meta['categories']))
print('last layer:', pretrained.fc)

<div dir="rtl">

## ۳. دیتاست: Oxford Flowers-102

**Flowers-102** (Nilsback & Zisserman, 2008) شامل تصاویر ۱۰۲ گونه گل است. نسخه رسمی فقط **۱۰ تصویر آموزشی برای هر کلاس** (۱۰۲۰ تصویر) دارد، به‌علاوه ۱۰۲۰ تصویر اعتبارسنجی و ۶۱۴۹ تصویر آزمون. این شرایط «داده کم و کلاس زیاد» دقیقاً جایی است که یادگیری انتقالی می‌درخشد.

برای آموزش از Augmentation استاندارد ImageNet (`RandomResizedCrop` و قرینه افقی) و برای ارزیابی از پیش‌پردازش رسمی مدل استفاده می‌کنیم:

</div>

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = preprocess                    # resize 256 -> center crop 224 -> normalize

train_set = datasets.Flowers102('data', split='train', download=True, transform=train_tf)
val_set = datasets.Flowers102('data', split='val', download=True, transform=eval_tf)
test_set = datasets.Flowers102('data', split='test', download=True, transform=eval_tf)
print('train / val / test:', len(train_set), len(val_set), len(test_set))

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=128, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=128, num_workers=2, pin_memory=True)

show = datasets.Flowers102('data', split='train', download=True)
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for ax, i in zip(axes.flat, range(0, 120, 10)):
    img, lab = show[i]
    ax.imshow(img); ax.set_title(f'class {lab}', fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

<div dir="rtl">

قبل از هر آموزشی، ببینیم مدل ImageNet درباره این تصاویر چه فکری می‌کند. ImageNet کلاس‌هایی مثل *daisy* دارد، اما بیشتر گونه‌های Flowers-102 در آن نیستند:

</div>

In [ ]:
@torch.no_grad()
def imagenet_top5(img):
    probs = torch.softmax(pretrained(preprocess(img).unsqueeze(0).to(device)), dim=1)[0]
    top = probs.topk(5)
    return [(weights.meta['categories'][i], p.item()) for p, i in zip(top.values, top.indices)]


for i in (0, 50):
    img, lab = show[i]
    print(f'Flowers-102 class {lab}:', ', '.join(f'{c} ({p:.2f})' for c, p in imagenet_top5(img)))

<div dir="rtl">

## ۴. توابع آموزش و Mixed Precision

برای سرعت بیشتر روی GPU از **Mixed Precision** استفاده می‌کنیم: بیشتر محاسبات با دقت ۱۶ بیتی (`float16`) انجام می‌شود که روی Tensor Coreهای GPU چند برابر سریع‌تر است و حافظه کمتری مصرف می‌کند، در حالی که وزن‌ها با دقت ۳۲ بیتی نگه داشته می‌شوند. `torch.autocast` نوع داده هر عملیات را خودکار انتخاب می‌کند و `GradScaler` با بزرگ‌کردن موقت Loss از صفر شدن گرادیان‌های کوچک در `float16` جلوگیری می‌کند. روی CPU این بخش خودکار غیرفعال می‌شود.

</div>

In [ ]:
USE_AMP = device.type == 'cuda'


def train_one_epoch(model, loader, criterion, optimizer, scaler, scheduler=None):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(X)
            loss = criterion(logits, y)
        optimizer.zero_grad()
        scaler.scale(loss).backward()        # scaled backward to avoid float16 underflow
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(X)
            total_loss += criterion(logits, y).item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


def run_experiment(model, param_groups, epochs, label):
    """Train with AdamW + cosine schedule; keep the weights with the best validation accuracy."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(train_loader))
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    history, best_acc, best_state = {'train_acc': [], 'val_acc': []}, 0.0, None
    start = time.time()
    for epoch in range(epochs):
        _, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, scheduler)
        _, va_acc = evaluate(model, val_loader, criterion)
        history['train_acc'].append(tr_acc); history['val_acc'].append(va_acc)
        if va_acc > best_acc:
            best_acc, best_state = va_acc, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    print(f'{label}: best val acc {best_acc:.4f} | {time.time() - start:.0f}s')
    return model, history


N_CLASSES = 102
EPOCHS = 15
results = {}

<div dir="rtl">

در `CrossEntropyLoss` از **Label Smoothing** استفاده کردیم: به‌جای برچسب One-hot صددرصدی، کمی احتمال (اینجا ۱۰٪) بین بقیه کلاس‌ها پخش می‌شود تا مدل بیش از حد مطمئن نشود؛ یک منظم‌سازی ساده و مؤثر برای داده کم.

## ۵. راهبرد ۱: آموزش از صفر

ابتدا یک ResNet-18 با وزن‌های تصادفی را با ۱۰۲۰ تصویر آموزش می‌دهیم تا خط مبنا مشخص شود:

</div>

In [ ]:
torch.manual_seed(0)
scratch = models.resnet18(weights=None, num_classes=N_CLASSES)
scratch, results['from scratch'] = run_experiment(scratch, [{'params': scratch.parameters(), 'lr': 1e-3}], EPOCHS, 'from scratch')

<div dir="rtl">

همان‌طور که انتظار داشتیم، ۱۰ تصویر برای هر کلاس برای آموزش یک شبکه ۱۱ میلیون پارامتری از صفر کافی نیست.

## ۶. راهبرد ۲: استخراج ویژگی (Feature Extraction)

همه وزن‌های مدل ازپیش‌آموزش‌دیده را **منجمد** می‌کنیم (`requires_grad = False`) و فقط لایه `fc` آخر را با یک لایه جدید با ۱۰۲ خروجی جایگزین می‌کنیم. لایه جدید به‌طور پیش‌فرض `requires_grad=True` دارد؛ پس فقط آن آموزش می‌بیند. این کار بسیار سریع است چون Backpropagation فقط تا لایه آخر انجام می‌شود.

</div>

In [ ]:
torch.manual_seed(0)
feat = models.resnet18(weights=weights)
for p in feat.parameters():
    p.requires_grad = False                          # freeze the backbone
feat.fc = nn.Linear(feat.fc.in_features, N_CLASSES)  # new, trainable classification head

trainable = sum(p.numel() for p in feat.parameters() if p.requires_grad)
total = sum(p.numel() for p in feat.parameters())
print(f'trainable parameters: {trainable:,} of {total:,} ({trainable / total:.2%})')
feat, results['feature extraction'] = run_experiment(feat, [{'params': feat.fc.parameters(), 'lr': 1e-3}], EPOCHS, 'feature extraction')

<div dir="rtl">

**نکته درباره BatchNorm:** وقتی Backbone منجمد است، `model.train()` باز هم آمار در حال اجرای (Running Statistics) لایه‌های BatchNorm را با داده جدید به‌روز می‌کند. با داده بسیار کم بهتر است لایه‌های BatchNorm منجمد را در حالت `eval` نگه دارید (تمرین ۳).

## ۷. راهبرد ۳: Fine-tuning کامل

حالا همه لایه‌ها را آموزش می‌دهیم، اما با **نرخ یادگیری متفاوت (Discriminative Learning Rates)**: نرخ کوچک برای Backbone تا ویژگی‌های ارزشمند پیش‌آموزش از بین نروند، و نرخ بزرگ‌تر برای لایه جدید که از صفر شروع می‌کند. PyTorch این کار را با **گروه‌های پارامتر (Parameter Groups)** در Optimizer انجام می‌دهد:

</div>

In [ ]:
torch.manual_seed(0)
ft = models.resnet18(weights=weights)
ft.fc = nn.Linear(ft.fc.in_features, N_CLASSES)
backbone_params = [p for name, p in ft.named_parameters() if not name.startswith('fc.')]
param_groups = [
    {'params': backbone_params, 'lr': 1e-4},        # small steps for pretrained layers
    {'params': ft.fc.parameters(), 'lr': 1e-3},     # larger steps for the new head
]
ft, results['fine-tuning'] = run_experiment(ft, param_groups, EPOCHS, 'fine-tuning')

In [ ]:
plt.figure(figsize=(9, 4.5))
for name, h in results.items():
    plt.plot(range(1, EPOCHS + 1), h['val_acc'], 'o-', label=name)
plt.xlabel('epoch'); plt.ylabel('validation accuracy'); plt.title('Flowers-102 with 10 training images per class')
plt.legend(); plt.show()

criterion = nn.CrossEntropyLoss()
for name, model in [('from scratch', scratch), ('feature extraction', feat), ('fine-tuning', ft)]:
    print(f'{name:20s} test accuracy: {evaluate(model, test_loader, criterion)[1]:.4f}')

<div dir="rtl">

**تحلیل:** با همان ۱۰۲۰ تصویر، آموزش از صفر به دقت پایینی می‌رسد، اما یادگیری انتقالی دقت را چند برابر می‌کند. Fine-tuning کامل معمولاً کمی بهتر از Feature Extraction است، به قیمت محاسبات بیشتر و ریسک بیشتر Overfitting.

### پیش‌بینی‌های مدل

</div>

In [ ]:
@torch.no_grad()
def show_predictions(model, dataset, raw_dataset, n=8):
    model.eval()
    idx = np.random.default_rng(1).choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 3))
    for ax, i in zip(axes, idx):
        x, y = dataset[i]
        pred = model(x.unsqueeze(0).to(device)).argmax(1).item()
        ax.imshow(raw_dataset[i][0]); ax.axis('off')
        ax.set_title(f'true {y}\npred {pred}', color='green' if pred == y else 'crimson', fontsize=10)
    plt.tight_layout(); plt.show()


show_predictions(ft, test_set, datasets.Flowers102('data', split='test', download=True))
torch.save(ft.state_dict(), 'flowers_resnet18.pt')

<div dir="rtl">

## ۸. راهنمای عملی

- **همیشه از پیش‌پردازش مدل پیش‌آموزش‌دیده استفاده کنید** (`weights.transforms()` یا همان میانگین و انحراف معیار ImageNet).
- با **Feature Extraction** شروع کنید: سریع است و خط مبنای خوبی می‌دهد. اگر داده کافی دارید سراغ **Fine-tuning** بروید.
- در Fine-tuning نرخ یادگیری Backbone را ۱۰ تا ۱۰۰ برابر کوچک‌تر از لایه جدید بگیرید. روش دیگر: ابتدا چند Epoch فقط Head را آموزش دهید و سپس همه را Unfreeze کنید (**Gradual Unfreezing**).
- مدل بزرگ‌تر لزوماً بهتر نیست: با داده کم، مدل‌های کوچک‌تر مثل ResNet-18 یا EfficientNet-B0 اغلب کافی‌اند.
- منابع مدل‌های ازپیش‌آموزش‌دیده: `torchvision.models`، کتابخانه `timm` (صدها مدل بینایی) و Hugging Face Hub.

## ۹. جمع‌بندی

- یادگیری انتقالی دانش مدل آموزش‌دیده روی داده بزرگ را به مسئله‌ای با داده کم منتقل می‌کند.
- Feature Extraction: منجمد کردن Backbone و آموزش Head جدید. Fine-tuning: آموزش همه لایه‌ها با نرخ یادگیری کوچک.
- Parameter Groups امکان نرخ یادگیری متفاوت برای بخش‌های مختلف مدل را می‌دهد.
- Mixed Precision با `torch.autocast` و `GradScaler` آموزش روی GPU را سریع‌تر و کم‌حافظه‌تر می‌کند.
- Label Smoothing یک منظم‌سازی ساده برای داده کم است.

## ۱۰. تمرین

۱. به‌جای ResNet-18 از `efficientnet_b0` و `resnet50` استفاده کنید (نام لایه آخر EfficientNet `classifier` است). دقت، زمان آموزش و تعداد پارامتر را مقایسه کنید.

۲. **Gradual Unfreezing** را پیاده کنید: ۵ Epoch فقط `fc`، سپس `layer4` را هم آزاد کنید، و در ۵ Epoch آخر همه لایه‌ها را.

۳. در حالت Feature Extraction، یک بار لایه‌های BatchNorm را در حالت `eval` نگه دارید (بعد از `model.train()` روی ماژول‌های `nn.BatchNorm2d` متد `eval()` را صدا بزنید) و نتیجه را مقایسه کنید.

۴. با ۴۰ تصویر دلخواه از دو یا سه دسته (مثلاً عکس‌های گوشی خودتان) یک Dataset با `datasets.ImageFolder` بسازید و یک طبقه‌بند با Fine-tuning آموزش دهید.

۵. دقت را بدون Label Smoothing (`label_smoothing=0`) تکرار کنید.

</div>